# Projeto 3 — Interpolação e Regressão Polinomial
Este notebook implementa os conceitos da questão 1 para as questões 2 e 3 do Projeto 3 em Julia.
Ele mostra a matriz de avaliação em base canônica (Vandermonde), a base de Lagrange, e a matriz de regressão por mínimos quadrados.

## 1. Funções auxiliares
Definimos a matriz de Vandermonde para a base canônica e uma função para construir a matriz de avaliação na base de Lagrange.

In [1]:
using LinearAlgebra

function vandermonde(x::Vector{Float64}, d::Integer)
    m = length(x)
    V = zeros(Float64, m, d + 1)
    for i in 1:m, j in 1:d + 1
        V[i, j] = x[i]^(j - 1)
    end
    return V
end

function least_squares_matrix(A::AbstractMatrix{Float64})
    # A é a matriz de avaliação L_X em alguma base de P_d.
    # A solução dos mínimos quadrados seria p = R_X,d * b, onde R_X,d = pinv(A).
    return pinv(A)
end

function lagrange_basis(nodes::Vector{Float64})
    n = length(nodes)
    basis = Vector{Function}(undef, n)
    for i in 1:n
        xi = nodes[i]
        denom = prod(xi - nodes[j] for j in 1:n if j != i)
        basis[i] = x -> begin
            num = prod(x - nodes[j] for j in 1:n if j != i)
            return num / denom
        end
    end
    return basis
end

function lagrange_eval_matrix(X::Vector{Float64}, nodes::Vector{Float64})
    basis = lagrange_basis(nodes)
    m = length(X)
    k = length(nodes)
    M = zeros(Float64, m, k)
    for i in 1:m, j in 1:k
        M[i, j] = basis[j](X[i])
    end
    return M
end

lagrange_eval_matrix (generic function with 1 method)

## 2. Montando as matrizes de avaliação e regressão
Aqui simulamos um conjunto de pontos distintos `X` e construímos as matrizes `L_X` e `R_X,d` em duas bases diferentes.

In [9]:
X = collect(range(-1.0, 1.0, length = 200))
d = 10
A = vandermonde(X, d)
R = least_squares_matrix(A)
println("Matriz L_X em base canônica (Vandermonde): tamanho = ", size(A))
println("Matriz R_X,d em base canônica: tamanho = ", size(R))
println("cond(L_X) = ", cond(A))
println("cond(R_X,d) = ", cond(R))

Matriz L_X em base canônica (Vandermonde): tamanho = (200, 11)
Matriz R_X,d em base canônica: tamanho = (11, 200)
cond(L_X) = 2988.1042180993554
cond(R_X,d) = 2988.1042180994978


## 3. Base de Lagrange
Construímos a base de Lagrange usando os primeiros `d+1` pontos de `X`.

In [10]:
nodes = collect(range(-1.0, 1.0, length = d + 1))
A_lagr = lagrange_eval_matrix(X, nodes)
R_lagr = least_squares_matrix(A_lagr)
println("Matriz L_X em base de Lagrange: tamanho = ", size(A_lagr))
println("cond(L_X) em Lagrange = ", cond(A_lagr))
println("cond(R_X,d) em Lagrange = ", cond(R_lagr))

Matriz L_X em base de Lagrange: tamanho = (200, 11)
cond(L_X) em Lagrange = 34.412728885431044
cond(R_X,d) em Lagrange = 34.41272888543106


## 4. Teste com um vetor de dados
A melhor forma de ver o efeito do condicionamento é aplicar as matrizes a um vetor de observações.

In [4]:
b = sin.(2π .* X) .+ 0.1 .* randn(length(X))
p_mono = R * b
p_lagr = R_lagr * b
println("Erro residual usando base canônica: ", norm(A * p_mono - b))
println("Erro residual usando base de Lagrange: ", norm(A_lagr * p_lagr - b))

Erro residual usando base canônica: 0.8986807165813986
Erro residual usando base de Lagrange: 0.8986807165813977


## 5. Interpretação e próximos passos
- `L_X` em base canônica é a matriz de Vandermonde.
- `R_X,d` é a pseudo-inversa de `L_X`, que representa a regressão de mínimos quadrados.
- A base de Lagrange muda a forma de `L_X`, e isso pode afetar o condicionamento.

### Para avançar
1. Varie `m` e `d` e veja como `cond(A)` e `cond(R)` mudam.
2. Compare o comportamento usando `X` equiespaçados e `X` de Chebyshev.
3. Adicione um gráfico com `Plots` para mostrar o crescimento do condicionamento com `d`.

# Questão 2
### (a)
Tome $q \in \mathcal{P}_d$ qualquer. Daí,


$$R_{X,d}(L_X(q)) = \arg\min_{p \in \mathcal{P}_d} ||L_X(p) - L_X(q)||_2^2$$

É fato que para $p = q \implies ||L_X(p) - L_X(q)||_2^2 = 0$. Para garantir que $R_{X,d}(L_X(q)) = q$, e que $q$ é o único que satisfaz $\arg\min_{p \in \mathcal{P}_d} ||L_X(p) - L_X(q)||_2^2 = 0$, verificamos que $L_X$ é injetora, ou seja, $L_X(p) = L_X(q) \implies p = q$.

Para tal, vamos mostrar que $\dim(\mathcal{N}(L_X)) = 0$. Suponha $p \in \mathcal{N}(L_X)$. Então $L_X(p) = (0, \dots, 0)$, ou seja, $p(x_i) = 0$ para todo $x_i \in X$.

Como $p$ tem grau no máximo $d$, $p$ deve possuir no máximo $d$ raízes. Mas $p(x) = 0$ para $m > d$ diferentes valores de $x$, logo, $p$ só pode ser o polinômio identicamente nulo. Ou seja, $\dim(\mathcal{N}(L_X)) = 0$ e $L_X$ é injetiva, como queríamos mostrar.


### (b)

Primeiramente, aproveitando os resultados da Questão 1, podemos expressar a transformação $L_X$ sobre $p \in \mathcal{P}_d$ na forma matricial como $L_X(p) = Vc$, onde $V$ é a matriz de Vandermonde de dimensões $m \times (d+1)$ dos pontos do conjunto $X$, e $c$ é o vetor com os coeficientes de $p$.

Desse modo, o problema de encontrar o polinômio $p$ equivale a encontrar os coeficientes $c$ que minimizam a expressão:
$$\arg\min_{c \in \mathbb{R}^{d+1}} ||Vc - b||_2^2$$

Note que:
$$||Vc-b||_2^2 = (Vc-b)^T(Vc-b) = c^T V^T V c - c^T V^T b - b^T V c + b^T b = c^T V^T V c - 2b^T V c + b^T b$$

Para encontrar o vetor $c$ que minimiza essa função, calculamos o gradiente em relação a $c$ e igualamos ao vetor nulo:
$$\nabla_c (c^T V^T V c - 2b^T V c + b^T b) = 2V^TVc - 2V^Tb = 0$$

Como $m \ge d+1$ e os pontos de $X$ são distintos, $V$ tem posto completo nas colunas, o que garante que $V^TV$ seja inversível. Isolando $c$:
$$V^TVc = V^Tb \implies c = (V^TV)^{-1}V^Tb$$

Assim, transformação $R_{X,d}$ mapeia o vetor $b$ para os coeficientes do polinômio essencialmente através da multiplicação matricial $(V^TV)^{-1}V^Tb$, logo vale a linearidade:

1. $R_{X,d}(b_1 + b_2) = (V^TV)^{-1}V(b_1 + b_2) = (V^TV)^{-1}Vb_1 + (V^TV)^{-1}Vb_2 = R_{X,d}(b_1) + R_{X,d}(b_2)$
2. $R_{X,d}(\alpha b) = (V^TV)^{-1}V(\alpha b) = \alpha((V^TV)^{-1}Vb) = \alpha R_{X,d}(b)$

Logo, concluímos que $R_{X,d}$ é uma transformação linear.

### (c)

Do item anterior, a matriz de $R_{X,d}$ na base canônica é a própria $(V^TV)^{-1}V^T$, onde $V$ é construída avaliando a base canônica $\{1, x, x^2, \dots, x^d\}$ nos pontos $x \in X$.

### (d)

### (e)

### (f)

### (g)

# Questão 3